# **Conversational RAG Application** with LangChain and Gemini LLM

In [1]:
# Install the necessary packages

!pip install langchain -qU
# !pip install langchain-openai -qU
!pip install langchain-chroma -qU
!pip install langchain_community -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220

In [7]:
!pip install -qU langchain_community
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00


In [8]:
import os
from google.colab import userdata

### Initialize Gemini LLM

In [16]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Set Gemini API key
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0
)

# # Example call
# response = llm.invoke("Hello! What is sri lanka?")
# print(response)

content=[{'type': 'text', 'text': 'Sri Lanka is an island nation located in **South Asia**, situated in the Indian Ocean just southeast of the tip of India. It is often referred to as the "Pearl of the Indian Ocean" because of its incredible natural beauty and its shape.\n\nHere is a breakdown of what makes Sri Lanka unique:\n\n### 1. Geography and Nature\nSri Lanka is famous for its diverse landscapes. Despite being a relatively small island, it features:\n*   **Golden Beaches:** Coastal areas like Mirissa, Unawatuna, and Arugam Bay are world-famous for surfing and relaxation.\n*   **The Hill Country:** The central part of the island is mountainous and cool, home to lush tea plantations and waterfalls (cities like Ella and Nuwara Eliya).\n*   **Wildlife:** It is one of the best places in Asia to see animals in the wild, including elephants, leopards, and blue whales.\n\n### 2. History and Heritage\nSri Lanka has a documented history spanning over 3,000 years.\n*   **Ancient Kingdoms:*

### Initialize Embedding Model

In [15]:
import os
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

# Set your Gemini (Google) API key
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Initialize Gemini embeddings
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"  # recommended base embedding model
)


### Load PDF Document

In [23]:
!pip install -qU pypdf langchain langchain-google-genai chromadb

In [29]:
from langchain_community.document_loaders import PyPDFLoader

# PDF එක load කිරීම
loader = PyPDFLoader("/content/Letter.pdf")
docs = loader.load()
print(f"📄 Total pages loaded: {len(docs)}")

📄 Total pages loaded: 1


In [30]:
len(docs)

1

### Split Documents into Chunks

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text splitter initialize කිරීම
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# PDF docs split කිරීම
splits = text_splitter.split_documents(docs)
print(f"✂️ Total chunks: {len(splits)}")

✂️ Total chunks: 4


### Create Vector Store and Retriever

In [35]:
from langchain_chroma import Chroma
import time

# Batch size set කිරීම quota limits වලට
batch_size = 10
all_vectors = []
success = False

for i in range(0, len(splits), batch_size):
    batch = splits[i:i+batch_size]
    while not success:
        try:
            vectors = embedding_model.embed_documents(batch)
            all_vectors.extend(vectors)
            success = True
        except Exception as e:
            print("⚠️ Quota limit hit, 20 seconds wait…")
            time.sleep(20)
    success = False  # reset for next batch

# Chroma vector store එකට insert කිරීම
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    collection_name="AgstatBK",
    persist_directory="./chroma_db"
)

print("✅ Vector store creation complete!")

⚠️ Quota limit hit, 20 seconds wait…
⚠️ Quota limit hit, 20 seconds wait…


KeyboardInterrupt: 

In [33]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

NameError: name 'vectorstore' is not defined

### Define Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define the system prompt
system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

# Create the prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])

### Create Retrieval-Augmented Generation (RAG) Chain

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the RAG chain
rag_chain = create_retrieval_chain(retriever, qa_chain)

### Invoke RAG Chain with Example Questions

In [ ]:
response = rag_chain.invoke({"input": "who is codeprolk"})
response["answer"]

'CodePRO LK is a dynamic educational platform founded by Dinesh Piyasamara during the COVID-19 pandemic. It offers a diverse range of technology-related courses in Sinhala, focusing on programming, data science, and machine learning. CodePRO LK aims to empower Sri Lankans with valuable skills in the tech industry through accessible and high-quality education. The platform continues to evolve and expand its offerings to support its mission of preparing learners for success in the global tech industry.'

In [ ]:
response = rag_chain.invoke({"input": "what is rag architecture"})
response["answer"]

'I\'m not familiar with "rag architecture." It seems to be a term or concept that is not related to the context provided about CodePRO LK and its community engagement, partnerships, and future directions in tech education. If you have more context or details about "rag architecture," I may be able to provide a better answer.'

In [ ]:
response = rag_chain.invoke({"input": "what are the courses codeprolk offer"})
response["answer"]

'CodePRO LK offers a variety of free courses presented in Sinhala, catering to various proficiency levels from beginners to intermediates. Some key courses include Python GUI – Tkinter. In the future, CodePRO LK aims to expand its offerings to cover advanced topics like artificial intelligence, cybersecurity, and advanced data analytics.'

In [ ]:
response = rag_chain.invoke({"input": "can you list down"})
response["answer"]

'List down what specifically? Please provide more details so I can assist you accurately.'

## Add Chat History

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# Define the contextualize system prompt
contextualize_system_prompt = (
    "using chat history and the latest user question, just reformulate question if needed and otherwise return it as is"
)

# Create the contextualize prompt template
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create the history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

#### Create History-Aware RAG Chain

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

prompt

ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know.\n\n{context}")), MessagesPlaceholder(variable_name='chat_history'), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])

In [ ]:
# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the history aware RAG chain
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

#### Manage Chat Session History

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Initialize the store for session histories
store = {}

# Function to get the session history for a given session ID
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create the conversational RAG chain with session history
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

#### Invoke Conversational RAG Chain with Example Questions

In [ ]:
response = conversational_rag_chain.invoke(
    {"input": "who is codeprolk"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'CodePRO LK is a dynamic educational platform founded by Dinesh Piyasamara during the COVID-19 pandemic. It offers a diverse range of technology-related courses in Sinhala, focusing on programming, data science, and machine learning. CodePRO LK aims to empower Sri Lankans with valuable skills in the tech industry through accessible and high-quality education. The platform continues to evolve and expand its offerings to support its mission of preparing learners for success in the global tech industry. Additionally, CodePRO LK engages its community through various events like webinars, live coding sessions, hackathons, and tech talks to provide networking opportunities and practical experience. The platform also collaborates with educational institutions, tech companies, and industry experts to enhance its content and resources, ensuring learners are well-prepared for real-world challenges.'

In [ ]:
store

{'101': InMemoryChatMessageHistory(messages=[HumanMessage(content='who is codeprolk'), AIMessage(content='CodePRO LK is a dynamic educational platform founded by Dinesh Piyasamara during the COVID-19 pandemic. It offers a diverse range of technology-related courses in Sinhala, focusing on programming, data science, and machine learning. CodePRO LK aims to empower Sri Lankans with valuable skills in the tech industry through accessible and high-quality education. The platform continues to evolve and expand its offerings to support its mission of preparing learners for success in the global tech industry. Additionally, CodePRO LK engages its community through various events like webinars, live coding sessions, hackathons, and tech talks to provide networking opportunities and practical experience. The platform also collaborates with educational institutions, tech companies, and industry experts to enhance its content and resources, ensuring learners are well-prepared for real-world chall

In [ ]:
response = conversational_rag_chain.invoke(
    {"input": "what is rag architecture"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'I\'m sorry, but I don\'t have information on "rag architecture." It seems to be a specific term or concept that is not related to the context provided about CodePRO LK. If you have any other questions or need clarification on a different topic, feel free to ask!'

In [ ]:
response = conversational_rag_chain.invoke(
    {"input": "what are the courses codeprolk offer"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

"CodePRO LK offers a variety of technology-related courses in Sinhala to empower learners with valuable skills in programming, data science, and machine learning. Some of the key courses offered by CodePRO LK include:\n\n1. Python GUI – Tkinter: This course covers the essentials of creating graphical user interfaces using Python's Tkinter library.\n\nCodePRO LK aims to cater to learners of all proficiency levels, from beginners to intermediates, ensuring that individuals at different stages can benefit from the courses. Additionally, the platform plans to expand its course offerings in the future to cover more advanced topics and emerging technologies such as artificial intelligence, cybersecurity, and advanced data analytics."

In [ ]:
response = conversational_rag_chain.invoke(
    {"input": "can you list down"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

"I apologize for the confusion earlier. Here is a list of some of the courses offered by CodePRO LK:\n\n1. Python GUI – Tkinter: This course covers the essentials of creating graphical user interfaces using Python's Tkinter library.\n\nPlease note that CodePRO LK aims to expand its course offerings in the future to cover more advanced topics and emerging technologies such as artificial intelligence, cybersecurity, and advanced data analytics."